# MVA v3 · NB04A — preregistered bulk-WGS clonal mCA screen
**VCF-only · BAF-primary · DP-secondary · privacy-preserving**

## Scientific scope

This notebook does **not** claim to measure total cell-by-cell mosaic-variegated aneuploidy.

Bulk WGS can detect a chromosome-scale mosaic event only when enough cells share a coherent allelic/copy-number imbalance.
Random gains/losses of different homologs across different cells can cancel in bulk DNA.

Therefore the output is:

> **detectable clonal chromosome-scale allelic imbalance / mCA burden**

not:

> total fraction of all aneuploid cells.

A negative result means only that no coherent chromosome-scale event above the preregistered detection threshold was detected.

## Frozen call threshold (before viewing BAF results)

Autosomes only.

A chromosome is called `BAF_MCA_CANDIDATE` only if all apply:

1. at least **200** informative heterozygous biallelic SNPs after QC;
2. fitted symmetric BAF split `delta_hat >= 0.020`;
3. bootstrap 95% lower CI for `delta_hat >= 0.015`;
4. likelihood-ratio support over the no-imbalance model;
5. signal is not explained by a genome-wide shift shared by most autosomes;
6. DP is reported only as secondary support because this VCF is variant-only (`no-ALT records = 0`).

`delta = 0.020` corresponds approximately to:
- ~8.3% mosaic fraction for a one-copy gain under a simple single-clone model;
- ~7.7% mosaic fraction for a one-copy loss.

Those percentages are model-equivalent fractions, not direct cell counts.

## QC

- FILTER must be PASS or "."
- biallelic SNP only
- GT simple heterozygous 0/1 or 1/0
- GQ >= 30
- DP 10–100
- each allele depth >= 5
- autosomes 1–22
- retain at most one marker per 1 kb to reduce local correlation

No raw variant rows are exported.

In [ ]:
import os
# Dependency bootstrap
import importlib.util, subprocess, sys
for pkg in ["cyvcf2","scipy"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])

from cyvcf2 import VCF
from scipy.optimize import minimize_scalar
from scipy.special import gammaln, logsumexp
import numpy as np, pandas as pd, json, math, os
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

ROOT = Path(os.environ.get("MVA_ROOT", "/path/to/MVA_Hackathon_2026"))
DATA_DIR = ROOT/"00_PRIVATE_DATA"/"hf_snapshot"
PRIVATE_OUT = ROOT/"02_RESULTS_PRIVATE"/"NB04A_BULK_MCA_SCREEN_v1_0_0"
PUBLIC_OUT = ROOT/"03_PUBLIC_ARTIFACTS"
PRIVATE_OUT.mkdir(parents=True, exist_ok=True)
PUBLIC_OUT.mkdir(parents=True, exist_ok=True)

vcfs = sorted(DATA_DIR.glob("*.vcf.gz"))
assert len(vcfs)==1, f"Expected one private VCF, found {len(vcfs)}"
VCF_IN=vcfs[0]
print("Private VCF located locally. Raw rows will not be exported.")

## 1. Extract high-quality heterozygous SNPs locally

In [ ]:
def fmt_scalar(var,key):
    try:
        x=var.format(key)
        if x is None or len(x)==0: return None
        x=x[0]
        if isinstance(x,np.ndarray):
            if x.size==1: return x.item()
            return x.tolist()
        if isinstance(x,np.generic): return x.item()
        return x
    except Exception:
        return None

rows=[]
last_pos={str(i):-10_000 for i in range(1,23)}
v=VCF(str(VCF_IN))

for var in v:
    chrom=str(var.CHROM).replace("chr","")
    if chrom not in last_pos:
        continue
    if var.FILTER not in (None,"PASS","."):
        continue
    if len(var.REF)!=1 or len(var.ALT)!=1 or len(var.ALT[0])!=1:
        continue

    gt=var.genotypes[0] if var.genotypes else None
    if gt is None or sorted(gt[:2]) != [0,1]:
        continue

    gq=fmt_scalar(var,"GQ")
    dp=fmt_scalar(var,"DP")
    ad=fmt_scalar(var,"AD")
    if gq is None or dp is None or ad is None:
        continue
    try:
        gq=float(gq); dp=int(dp)
        if isinstance(ad,(list,tuple,np.ndarray)):
            if len(ad)<2: continue
            refc=int(ad[0]); altc=int(ad[1])
        else:
            continue
    except Exception:
        continue

    if gq < 30 or dp < 10 or dp > 100 or refc < 5 or altc < 5:
        continue
    if var.POS - last_pos[chrom] < 1000:
        continue
    last_pos[chrom]=var.POS

    n=refc+altc
    baf=altc/n if n>0 else np.nan
    rows.append((chrom,int(var.POS),refc,altc,n,baf,dp,gq))

v.close()

het=pd.DataFrame(rows,columns=["chrom","pos","ref_count","alt_count","ad_total","baf","dp","gq"])
assert len(het)>0
het.to_parquet(PRIVATE_OUT/"hq_het_snps_private.parquet",index=False)

print("HQ heterozygous SNPs:",len(het))
print(het.groupby("chrom").size().to_string())

## 2. Symmetric BAF-mixture model

In [ ]:
# Unphased whole-chromosome gain/loss produces mirrored BAF modes:
# 0.5-delta and 0.5+delta. We fit a 50:50 symmetric mixture.
# This is intentionally conservative and does not classify gain vs loss.

def log_binom_pmf(k,n,p):
    return gammaln(n+1)-gammaln(k+1)-gammaln(n-k+1)+k*np.log(p)+(n-k)*np.log(1-p)

def ll_delta(delta, alt, total):
    p1=0.5-delta
    p2=0.5+delta
    if p1<=0 or p2>=1: return -np.inf
    a=log_binom_pmf(alt,total,p1)
    b=log_binom_pmf(alt,total,p2)
    return np.sum(logsumexp(np.vstack([a,b]),axis=0)-np.log(2.0))

def fit_delta(df):
    alt=df["alt_count"].to_numpy(dtype=int)
    total=df["ad_total"].to_numpy(dtype=int)
    ll0=ll_delta(0.0,alt,total)
    res=minimize_scalar(lambda d:-ll_delta(d,alt,total),bounds=(0,0.16),method="bounded")
    d=float(res.x)
    ll1=-float(res.fun)
    return d,ll0,ll1,2*(ll1-ll0)

def bootstrap_delta(df,B=300,seed=20260922):
    rng=np.random.default_rng(seed)
    vals=[]
    n=len(df)
    for _ in range(B):
        idx=rng.integers(0,n,n)
        d,*_=fit_delta(df.iloc[idx])
        vals.append(d)
    return np.quantile(vals,[0.025,0.5,0.975]), vals

summary=[]
boot_cache={}
for c in map(str,range(1,23)):
    z=het[het.chrom==c].copy()
    if len(z)<50:
        summary.append({"chrom":c,"n_markers":len(z),"delta_hat":np.nan,"lrt":np.nan,
                        "ci_low":np.nan,"ci_med":np.nan,"ci_high":np.nan})
        continue
    d,ll0,ll1,lrt=fit_delta(z)
    ci,vals=bootstrap_delta(z,B=300,seed=20260922+int(c))
    boot_cache[c]=vals
    summary.append({
        "chrom":c,"n_markers":len(z),"delta_hat":d,"lrt":lrt,
        "ci_low":float(ci[0]),"ci_med":float(ci[1]),"ci_high":float(ci[2]),
        "median_dp_variant_sites":float(z.dp.median())
    })

res=pd.DataFrame(summary)
display(res)

## 3. Fixed call rule + model-equivalent mosaic fraction

In [ ]:
# Model conversions from |BAF-0.5| = delta.
# single-clone trisomy: lower BAF = 1/(2+f)
# single-clone monosomy: lower BAF = (1-f)/(2-f)

def f_gain_from_delta(d):
    b=0.5-d
    return max(0.0, min(1.0, 1.0/b - 2.0))

def f_loss_from_delta(d):
    b=0.5-d
    # b(2-f)=1-f -> f=(1-2b)/(1-b)
    return max(0.0, min(1.0, (1.0-2.0*b)/(1.0-b)))

eligible=res.n_markers>=200
genome_bg=float(res.loc[eligible,"delta_hat"].median()) if eligible.any() else np.nan

def call_row(r):
    if r.n_markers < 200: return "INSUFFICIENT_MARKERS"
    if not (r.delta_hat >= 0.020 and r.ci_low >= 0.015): return "NO_CALL"
    # chromosome-specific excess over the autosomal background
    if np.isfinite(genome_bg) and (r.delta_hat-genome_bg)<0.010:
        return "GLOBAL_BIAS_HOLD"
    if r.lrt < 25:
        return "WEAK_LIKELIHOOD_HOLD"
    return "BAF_MCA_CANDIDATE"

res["call"]=res.apply(call_row,axis=1)
res["gain_equiv_fraction"]=res["delta_hat"].map(lambda x:f_gain_from_delta(x) if np.isfinite(x) else np.nan)
res["loss_equiv_fraction"]=res["delta_hat"].map(lambda x:f_loss_from_delta(x) if np.isfinite(x) else np.nan)
res["delta_minus_autosomal_median"]=res["delta_hat"]-genome_bg

display(res)
print("Autosomal median fitted delta:",genome_bg)
print(res["call"].value_counts())

## 4. DP as secondary sanity check only

In [ ]:
# This VCF has no no-ALT/reference blocks, so DP is conditioned on variant sites.
# Report a chromosome-normalized median DP ratio only as secondary evidence.
meddp=res.set_index("chrom")["median_dp_variant_sites"]
global_dp=float(np.nanmedian(meddp.values))
res["variant_site_dp_ratio"]=res["median_dp_variant_sites"]/global_dp
display(res[["chrom","n_markers","delta_hat","ci_low","ci_high","variant_site_dp_ratio","call"]])

## 5. Sanitized handoff

In [ ]:
from datetime import datetime, timezone

public_cols=[
    "chrom","n_markers","delta_hat","ci_low","ci_high","lrt",
    "gain_equiv_fraction","loss_equiv_fraction",
    "variant_site_dp_ratio","call"
]
public_df=res[public_cols].copy()
csv_out=PUBLIC_OUT/"NB04A_MCA_CHROMOSOME_SUMMARY.csv"
public_df.to_csv(csv_out,index=False)

handoff={
    "schema":"MVA-NB04A-BULK-MCA-SCREEN-1.0",
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "scope":"bulk-WGS clonal chromosome-scale allelic-imbalance screen; NOT total variegated aneuploid-cell fraction",
    "preregistered_threshold":{
        "min_markers":200,
        "delta_hat_min":0.020,
        "bootstrap_ci_low_min":0.015,
        "lrt_min":25,
        "chromosome_excess_over_autosomal_median_min":0.010
    },
    "approx_detection_equivalent":{
        "single_clone_gain_fraction_at_delta_0_02":f_gain_from_delta(0.020),
        "single_clone_loss_fraction_at_delta_0_02":f_loss_from_delta(0.020)
    },
    "primary_signal":"AD-derived BAF symmetric mixture",
    "coverage_signal":"variant-site DP only; secondary because input is variant-only VCF",
    "autosomal_background_delta":genome_bg,
    "n_candidate_chromosomes":int((res.call=="BAF_MCA_CANDIDATE").sum()),
    "candidate_chromosomes":res.loc[res.call=="BAF_MCA_CANDIDATE","chrom"].tolist(),
    "negative_result_interpretation":"No coherent clonal chromosome-scale allelic imbalance above the preregistered threshold; does not exclude cell-to-cell variegated aneuploidy.",
    "privacy":{
        "raw_variant_rows_public":False,
        "genotypes_public":False,
        "sample_name_public":False,
        "only_chromosome_level_summary_public":True
    }
}
json_out=PUBLIC_OUT/"NB04A_MCA_HANDOFF.json"
json_out.write_text(json.dumps(handoff,indent=2))

print(json.dumps(handoff,indent=2))
print("Saved:",csv_out)
print("Saved:",json_out)